# Tratamento gold — Estações (Parquet)

Pega o inventário de estações já limpo pela silver (`estacoes.csv`) e
grava em Parquet, com cada coluna no tipo certo.

Entrada (silver): `data/silver/ana/estacoes/estacoes.csv`
Saída (gold):     `data/gold/ana/estacoes/estacoes.parquet`

Essa tabela é pequena (poucas colunas, ~4 mil linhas) — o ganho de
tamanho/velocidade do Parquet aqui é menor que em chuva/cota, mas manter o
mesmo formato deixa mais fácil de relacionar (fazer JOIN) as três tabelas
dentro do Power BI.

In [ ]:
from pathlib import Path

import pandas as pd


def encontrar_raiz_do_projeto(marcador: str = "pyproject.toml") -> Path:
    atual = Path.cwd().resolve()
    for pasta in [atual, *atual.parents]:
        if (pasta / marcador).exists():
            return pasta
    raise FileNotFoundError(
        f"Não encontrei '{marcador}' subindo a partir de {atual}. "
        "Este notebook está dentro da pasta do projeto?"
    )


RAIZ = encontrar_raiz_do_projeto()

ENTRADA_CSV = RAIZ / "data" / "silver" / "ana" / "estacoes" / "estacoes.csv"
SAIDA_PARQUET = RAIZ / "data" / "gold" / "ana" / "estacoes" / "estacoes.parquet"

print("Raiz do projeto:", RAIZ)
print("Lendo de:", ENTRADA_CSV)
print("Vai salvar em:", SAIDA_PARQUET)

## 1. Ler o CSV da silver

In [ ]:
silver = pd.read_csv(ENTRADA_CSV)
print(f"{len(silver)} linha(s) lida(s) da silver.")
silver.dtypes

## 2. Aplicar o schema final (tipo de cada coluna)

In [ ]:
gold = silver.copy()
gold["codigo"] = pd.to_numeric(gold["codigo"], errors="coerce").astype("Int64")
gold["tipo"] = gold["tipo"].astype("category")
gold["nome"] = gold["nome"].astype(str)
gold["latitude"] = gold["latitude"].astype("float64")
gold["longitude"] = gold["longitude"].astype("float64")

gold.dtypes

## 3. Padronizar nomes de colunas (prefixos)

Adiciona colunas com o padrão `cod_`/`nm_`/`tp_`/`vl_`, sem apagar as
colunas originais. Aqui não tem nenhum código numérico escondido pra
traduzir — `tipo` já vem como texto (`Pluviometrica`/`Fluviometrica`)
direto da ANA, então só ganha o prefixo `tp_`, sem coluna `ds_` separada.

In [ ]:
gold["cod_estacao"] = gold["codigo"]
gold["nm_estacao"] = gold["nome"]
gold["tp_estacao"] = gold["tipo"]
gold["vl_latitude"] = gold["latitude"]
gold["vl_longitude"] = gold["longitude"]

gold.columns.tolist()

## 4. Conferência rápida

In [ ]:
print("Linhas:", len(gold))
print("Nulos por coluna:")
print(gold.isna().sum())
gold.head()

## 5. Salvar em Parquet (gold)

In [ ]:
SAIDA_PARQUET.parent.mkdir(parents=True, exist_ok=True)
gold.to_parquet(SAIDA_PARQUET, index=False, engine="pyarrow")

tamanho_csv_mb = ENTRADA_CSV.stat().st_size / 1e6
tamanho_parquet_mb = SAIDA_PARQUET.stat().st_size / 1e6
print(f"Salvo em {SAIDA_PARQUET}")
print(f"Tamanho: {tamanho_csv_mb:.2f} MB (csv) -> {tamanho_parquet_mb:.2f} MB (parquet)")